# DBS add-on — 55 Hz, 60 µs

Cells to paste into `hh_400E_100I_gmax_gates_rest_tuning`. Section 1 needs a small edit to your
equation strings; everything else is additive.

**Two things to keep straight from the start**

`I_dbs` is injected into the membrane equation but must **never** enter the LFP proxy. Your
proxy is built from `I_AMPA_ext`, `I_AMPA_rec`, `I_GABA_rec` only, so this is already correct —
just don't be tempted to add `I_dbs` to it later. Stimulation artifact belongs in a separate
recording-chain model, otherwise you've baked the artifact into the neural signal and the whole
question becomes unanswerable.

`DT = 0.02*ms` gives exactly **3 samples** per 60 µs pulse. That is the minimum that works. Do
not raise `DT` while stimulation is on.

**A tuning note before you run.** Your resting bump is currently at **35 Hz**. At `f_s = 55`, the
prominent locking ratios put the rhythm at 27.5 Hz (1:2) or 82.5 Hz (3:2). A 35 Hz rhythm sits
between them — 35/55 = 0.64, closest to 2:3, which is a weak higher-order tongue. So you may see
little or nothing at first, and that would be a correct result rather than a bug. If you want to
test the 1:2 hypothesis properly, first move the resting peak down toward 27–28 Hz by widening
`S_IE.delay` back toward `'3*ms + rand()*3*ms'` or raising `TAU_GABA_DECAY`.

## 1. Equation edit

In **both** `eqs_E` and `eqs_I`, add `+ I_dbs` to the current sum in `dv/dt`:

```
dv/dt = (
      G_NA*m**3*h*(E_NA-v)
    + G_K*n**4*(E_K-v)
    + G_L*(E_L-v)
    + I_AMPA_ext
    + I_AMPA_rec
    + I_GABA_rec
    + I_tonic
    + I_dbs                 # <-- add
    )/CM : volt
```

and add these two lines next to `I_tonic : amp`:

```
I_dbs = G_DBS * dbs(t) : amp
G_DBS : amp
```

`dbs(t)` is a dimensionless `TimedArray` in mA; `G_DBS` converts mA to amps per neuron and
carries the distance dependence.

In [ ]:
# ============================================================
# DBS parameters
# ============================================================
F_STIM       = 55.0          # Hz
PULSE_WIDTH  = 60e-6         # s   (3 samples at DT = 0.02 ms)
RECHARGE     = 4.0           # negative phase: RECHARGE*PULSE_WIDTH at -amp/RECHARGE
DBS_AMP      = 3.0           # mA

# Coupling. I_PER_MA is the current delivered to a neuron at the reference
# distance per mA of stimulation. 5 nA/mA puts 1-3 mA in the interesting range
# and 10 mA near depolarisation block; 25 nA/mA blocks the E cells by 2 mA and
# flattens the whole sweep.
I_PER_MA     = 5.0           # nA per mA at R_REF
R_REF        = 2000.0        # um
GAIN_E       = 0.3           # relative sensitivity of E
GAIN_I       = 1.0           # relative sensitivity of I (stimulation targets I,
                             # following Sermon et al. 2023)

# Electrode geometry. Monopole point source: V ~ I/(4*pi*sigma*r), so the
# per-neuron gain scales as R_REF/r. Uniform for now; replace R_E/R_I with
# per-neuron distances when you add positions.
R_E = 2000.0                 # um
R_I = 2000.0                 # um

## 2. Waveform

Charge-balanced biphasic pulse train, built in numpy at `DT` and handed to Brian2 as a
`TimedArray`.

Verified: the train has power only at harmonics of `F_STIM` (55, 110, 165, ...) and nothing at
27.5 or 82.5 Hz — 16 orders of magnitude down. So any subharmonic in the output comes from the
network, not from the stimulus. The `jitter` and `alternation` arguments deliberately break that,
and are the device-artifact hypotheses to test later; leave both at 0 for now.

In [ ]:
def build_dbs_wave(duration_s, f_stim=F_STIM, amp=DBS_AMP, pulse_width=PULSE_WIDTH,
                   recharge=RECHARGE, dt_s=None, schedule=None,
                   jitter=0.0, alternation=0.0, seed_val=0):
    """Charge-balanced biphasic pulse train in mA, sampled at dt_s.

    schedule    : callable t(array of onset times, s) -> amplitude multiplier.
                  Use `staircase` for the charting protocol.
    jitter      : SD of relative period noise (Duchet et al. dithering level).
    alternation : +-fraction pulse-to-pulse amplitude modulation. Non-zero makes
                  the stimulus itself period-2, i.e. puts real lines at f_s/2 and
                  3*f_s/2. Keep at 0 unless you are testing that hypothesis.
    """
    dt_s = dt_s if dt_s is not None else float(DT/second)
    n = int(round(duration_s/dt_s))
    w_pos = int(round(pulse_width/dt_s))
    w_neg = int(round(recharge*pulse_width/dt_s))
    if w_pos < 2:
        raise ValueError(f'DT = {dt_s*1e6:.0f} us cannot resolve a '
                         f'{pulse_width*1e6:.0f} us pulse')

    rng = np.random.default_rng(seed_val)
    k = int(duration_s*f_stim*(1 + 6*jitter)) + 16
    periods = (1 + rng.normal(0, jitter, k))/f_stim if jitter else np.full(k, 1.0/f_stim)
    periods = np.clip(periods, (w_pos + w_neg)*dt_s, None)
    onsets = np.cumsum(periods) - periods[0]
    onsets = onsets[onsets < duration_s - (w_pos + w_neg)*dt_s]

    amps = amp*(schedule(onsets) if schedule is not None else np.ones_like(onsets))
    if alternation:
        amps = amps*(1 + alternation*(-1.0)**np.arange(len(onsets)))

    wave = np.zeros(n, dtype=np.float32)
    for s, a in zip(np.round(onsets/dt_s).astype(int), amps):
        if a:
            wave[s:s + w_pos] += a
            wave[s + w_pos:s + w_pos + w_neg] -= a*w_pos/w_neg
    return wave, onsets, amps


def staircase(t, duration_s, a_max=10.0, n_steps=10, baseline_frac=0.1):
    """Clinical charting protocol: a flat 0 mA baseline, then equal steps to a_max."""
    t0 = baseline_frac*duration_s
    step = (duration_s - t0)/n_steps
    idx = np.floor((np.asarray(t) - t0)/step)
    levels = np.linspace(a_max/n_steps, a_max, n_steps)
    return np.where(idx < 0, 0.0, levels[np.clip(idx, 0, n_steps - 1).astype(int)])


def amp_of_time(onsets, amps, t_grid):
    """Delivered amplitude on an arbitrary time grid (for plotting / step splitting)."""
    return np.interp(t_grid, onsets, amps, left=0.0, right=amps[-1] if len(amps) else 0.0)

## 3. Wire it into the network

Build the waveform for the full `SIM_TIME`, put it in **both** namespaces, and set `G_DBS` after
the groups are created. Put these lines before `net = Network(...)`; nothing needs to be added to
`Network` itself, since a `TimedArray` is not a network object.

In [ ]:
DBS_WAVE, DBS_ONSETS, DBS_AMPS = build_dbs_wave(float(SIM_TIME/second))
dbs = TimedArray(DBS_WAVE, dt=DT)

namespace_E['dbs'] = dbs
namespace_I['dbs'] = dbs

# ... create E and I as before, then:
E.G_DBS = I_PER_MA*nA * GAIN_E * (R_REF/R_E)
I.G_DBS = I_PER_MA*nA * GAIN_I * (R_REF/R_I)

print(f'DBS: {F_STIM} Hz, {PULSE_WIDTH*1e6:.0f} us, {DBS_AMP} mA, '
      f'{len(DBS_ONSETS)} pulses over {float(SIM_TIME/second):.0f} s')
print(f'per-pulse charge into an E cell: '
      f'{float(E.G_DBS[0]/nA)*DBS_AMP*PULSE_WIDTH*1e6:.1f} fC '
      f'-> dV ~ {float(E.G_DBS[0]/nA)*DBS_AMP*PULSE_WIDTH*1e-9/200e-12*1e3:.1f} mV per pulse')

## 4. Analysis

Three measurements, in increasing order of how hard they are to fake.

**PSD with markers** at `f_s/2`, `f_s`, `3f_s/2`. Note `f_s = 55` is inside your recorded band,
so the stimulation frequency itself will show up as a neural response (1:1 following), not only
as artifact.

**Entrainment index** — Sermon et al.'s criterion: peak height at the target frequency over a
baseline interpolated from `[f-10, f-5]` and `[f+5, f+10]`. Above 3 counts as entrained.

**Rotation number** — Sermon et al. eq. (1). `R = 0.5` means the rhythm completes half a cycle
per pulse, i.e. 1:2 locking. Important caveat: `R ≈ f_rhythm/f_s` *always*, so the value alone
proves nothing. Locking is a **plateau** — `R` staying pinned at 0.5 while the endogenous
frequency is varied. Sweep `TAU_GABA_DECAY` at fixed amplitude to test that.

In [ ]:
def dbs_psd(x, fs, nperseg_s=4.0):
    x = signal.detrend(np.asarray(x))
    nper = int(min(len(x), nperseg_s*fs))
    return signal.welch(x, fs=fs, nperseg=nper, noverlap=nper//2)


def ent_index(f, pxx, f0, gap=5.0, width=5.0, search=1.5):
    """Peak/baseline ratio at f0 and where the peak actually sits."""
    side = (((f >= f0 - gap - width) & (f <= f0 - gap)) |
            ((f >= f0 + gap) & (f <= f0 + gap + width)))
    if side.sum() < 4:
        return np.nan, np.nan
    base = 10**np.polyval(np.polyfit(f[side], np.log10(pxx[side]), 1), f0)
    k = np.abs(f - f0) <= search
    return pxx[k].max()/base, f[k][np.argmax(pxx[k])]


def peak_bandwidth(f, pxx, f0, search=4.0, floor_db=3.0):
    """-3 dB width (Hz). A device line is one frequency bin wide; an entrained
    neural rhythm inherits the rhythm's phase diffusion and is several bins wide."""
    k = np.abs(f - f0) <= search
    if k.sum() < 3:
        return np.nan
    fk, pk = f[k], pxx[k]
    i = int(np.argmax(pk)); thr = pk[i]/10**(floor_db/10)
    lo, hi = i, i
    while lo > 0 and pk[lo] > thr: lo -= 1
    while hi < len(pk) - 1 and pk[hi] > thr: hi += 1
    return fk[hi] - fk[lo]


def rotation_number(x, fs, onsets, band, t0=0.0):
    """Sermon et al. eq. (1): (theta_N - theta_0)/(2*pi*N), Hilbert phase of the
    band-passed signal sampled at pulse times."""
    sos = signal.butter(4, band, btype='bandpass', fs=fs, output='sos')
    th = np.unwrap(np.angle(signal.hilbert(signal.sosfiltfilt(sos, np.asarray(x)))))
    t = np.arange(len(x))/fs
    pt = onsets[(onsets > max(t0, t[0])) & (onsets < t[-1])]
    if len(pt) < 50:
        return np.nan
    ph = np.interp(pt, t, th)
    return (ph[-1] - ph[0])/(2*np.pi*(len(ph) - 1))


def report_dbs(lfp, t, f_stim=F_STIM, band=(20, 45)):
    fs = 1.0/float(REC_DT/second)
    keep = t >= float(DISCARD/second)
    x = lfp[keep]
    f, pxx = dbs_psd(x, fs)
    print(f'{"target":>10} {"idx":>7} {"peak Hz":>9} {"BW Hz":>7}')
    for lab, f0 in [('f_s/2', f_stim/2), ('f_s', f_stim), ('3f_s/2', 1.5*f_stim)]:
        idx, fpk = ent_index(f, pxx, f0)
        print(f'{lab:>10} {idx:7.2f} {fpk:9.2f} {peak_bandwidth(f, pxx, f0):7.2f}')
    R = rotation_number(x, fs, DBS_ONSETS, band, t0=float(DISCARD/second))
    print(f'rotation number (band {band[0]}-{band[1]} Hz): R = {R:.3f}'
          f'   [0.5 = 1:2, 1.0 = 1:1, 1.5 = 3:2]')
    return f, pxx


def plot_dbs_psd(f, pxx, f_stim=F_STIM, fmax=120):
    k = (f > 2) & (f <= fmax)
    plt.figure(figsize=(10, 3.6))
    plt.semilogy(f[k], pxx[k], lw=0.8)
    for f0, c in [(f_stim/2, 'C1'), (f_stim, 'C3'), (1.5*f_stim, 'C1')]:
        plt.axvline(f0, color=c, ls='--', lw=1, alpha=0.8)
    plt.xlabel('Frequency (Hz)'); plt.ylabel('PSD')
    plt.title(f'LFP proxy during {f_stim} Hz DBS '
              f'(dashed: f_s/2, f_s, 3f_s/2)')
    plt.tight_layout(); plt.show()

In [ ]:
# after net.run(SIM_TIME)
t, lfp_total, lfp_rec, lfp_ext, lfp_exc, lfp_inh = compute_lfp_proxies()
print_basic_summary()
f, pxx = report_dbs(lfp_rec, t)
plot_dbs_psd(f, pxx)

## 5. Amplitude charting

The discriminating measurement. A genuine Arnold tongue has a **threshold and an upper
boundary** — 1:2 entrainment is lost when amplitude is pushed past the top of the tongue, which
Sermon et al. observed in all three patients. An artifact grows monotonically and never closes.

At 30 s per run this is ~20 min for 10 steps in runtime mode. Either accept that, or switch to
`set_device('cpp_standalone', build_on_run=True)`, which is roughly 50× faster here — with
`device.reinit(); device.activate(...)` at the top of each build.

In [ ]:
CHART_DUR   = 160.0          # s
CHART_STEPS = 10
CHART_MAX   = 10.0           # mA

def run_charting(duration_s=CHART_DUR, a_max=CHART_MAX, n_steps=CHART_STEPS):
    """Rebuild the network with a staircase waveform and run it. Assumes your
    build cell has been wrapped in a function or is re-executed with
    SIM_TIME = duration_s*second and DBS_WAVE from below."""
    sched = lambda tt: staircase(tt, duration_s, a_max=a_max, n_steps=n_steps)
    wave, onsets, amps = build_dbs_wave(duration_s, amp=1.0, schedule=sched)
    return wave, onsets, amps


def step_psds(x, fs, amp_t, skip_s=3.0, nperseg_s=4.0):
    """One PSD per amplitude plateau."""
    a = np.asarray(amp_t)
    b = np.concatenate(([0], np.flatnonzero(np.diff(a) != 0) + 1, [len(a)]))
    out = []
    for i in range(len(b) - 1):
        s, e = b[i] + int(skip_s*fs), b[i + 1]
        if e - s < nperseg_s*fs:
            continue
        f, pxx = dbs_psd(x[s:e], fs, nperseg_s)
        out.append(dict(amp=float(a[b[i]]), f=f, psd=pxx))
    return out


def plot_amplitude_profile(steps, f_stim=F_STIM):
    fig, ax = plt.subplots(1, 3, figsize=(12, 3.2), sharex=True)
    for j, (lab, f0) in enumerate([('f_s/2', f_stim/2), ('f_s', f_stim),
                                   ('3f_s/2', 1.5*f_stim)]):
        a = [s['amp'] for s in steps]
        idx = [ent_index(s['f'], s['psd'], f0)[0] for s in steps]
        bw = [peak_bandwidth(s['f'], s['psd'], f0) for s in steps]
        ax[j].semilogy(a, idx, 'o-', ms=4, label='entrainment index')
        ax[j].plot(a, bw, 's--', ms=3, alpha=0.6, label='-3 dB width (Hz)')
        ax[j].axhline(3, c='k', ls=':', lw=1)
        ax[j].set(xlabel='amplitude (mA)', title=f'{lab} = {f0:.1f} Hz')
    ax[0].set_ylabel('index / width'); ax[0].legend(fontsize=7)
    plt.suptitle('tongue closes at high amplitude; artifact does not', fontsize=10)
    plt.tight_layout(); plt.show()

## 6. Next

Once 55 Hz behaves, the decisive test is **80 Hz**. If a line at `f_s/2` moves rigidly to 40 Hz,
it is a property of the pulse train. If it does not appear at all, the 55 Hz result was a tongue
attached to your endogenous rhythm — because 40 Hz is off it.

Only after that is it worth adding the recording chain (artifact leakage, 450 Hz low pass,
250 Hz sampling without anti-aliasing). Worth knowing in advance: at 250 Hz sampling the 4th
harmonic of 55 Hz folds to exactly 30 Hz and the 5th to 25 Hz, both inside beta. Any beta
measurement taken during 55 Hz stimulation sits under two folded artifact lines.